# Graph Database Concepts

## Overview
A **graph database** stores data as nodes (entities) and relationships (edges), with both having arbitrary properties. This structure directly models real-world connected data — researchers collaborating, drugs treating diseases, institutions funding trials.

**Core graph elements:**
```
(r:Researcher {id:'R001', name:'Aroha Ngata'})    ← Node with label and properties
(r)-[:LEADS {since:'2023-01-01'}]->(t:Trial)       ← Directed, typed relationship
```

**Why graphs are different:**
In a relational database, connections between entities require foreign keys and JOINs. Each additional "hop" in a traversal (friends of friends of friends) requires another JOIN. In a graph database, traversal is a pointer follow — O(1) per hop regardless of graph size.

**Dataset:** Clinical research network — Researchers, Institutions, Trials, Diseases, Drugs.

---

In [1]:
import networkx as nx

# ── Build a knowledge graph: clinical research network ────────────
# Nodes: Researchers, Institutions, Papers, Trials, Diseases, Drugs
# This mirrors how Neo4j would store these as labelled nodes with properties

G = nx.MultiDiGraph()

# Researchers
for r_id, name, role in [
    ("R001", "Aroha Ngata",   "Principal Investigator"),
    ("R002", "Liam Chen",     "Biostatistician"),
    ("R003", "Fatima Rashid", "Clinical Coordinator"),
    ("R004", "James MacLeod", "Researcher"),
]:
    G.add_node(r_id, label="Researcher", name=name, role=role)

# Institutions
for i_id, name, country in [
    ("I001", "Dalhousie University",    "Canada"),
    ("I002", "University of Toronto",   "Canada"),
    ("I003", "McGill University",       "Canada"),
]:
    G.add_node(i_id, label="Institution", name=name, country=country)

# Diseases
for d_id, name, category in [
    ("D001", "Type 2 Diabetes",         "Metabolic"),
    ("D002", "Hypertension",            "Cardiovascular"),
    ("D003", "Heart Failure",           "Cardiovascular"),
]:
    G.add_node(d_id, label="Disease", name=name, category=category)

# Drugs
for dr_id, name, drug_class in [
    ("DR001", "Metformin",    "Biguanide"),
    ("DR002", "Lisinopril",   "ACE Inhibitor"),
    ("DR003", "Atorvastatin", "Statin"),
]:
    G.add_node(dr_id, label="Drug", name=name, drug_class=drug_class)

# Trials
for t_id, title, phase in [
    ("T001", "Cardio Alpha Trial", 2),
    ("T002", "Neuro Beta Trial",   3),
]:
    G.add_node(t_id, label="Trial", title=title, phase=phase)

# Relationships
edges = [
    # Researchers at institutions
    ("R001","I001","AFFILIATED_WITH", {"since":"2015"}),
    ("R002","I002","AFFILIATED_WITH", {"since":"2018"}),
    ("R003","I001","AFFILIATED_WITH", {"since":"2020"}),
    ("R004","I003","AFFILIATED_WITH", {"since":"2012"}),
    # Researchers leading/working on trials
    ("R001","T001","LEADS",           {"role":"PI"}),
    ("R002","T001","CONTRIBUTES_TO",  {"role":"Statistician"}),
    ("R003","T001","CONTRIBUTES_TO",  {"role":"Coordinator"}),
    ("R004","T002","LEADS",           {"role":"PI"}),
    ("R002","T002","CONTRIBUTES_TO",  {"role":"Statistician"}),
    # Trials studying diseases
    ("T001","D002","STUDIES",         {"primary":True}),
    ("T001","D001","STUDIES",         {"primary":False}),
    ("T002","D003","STUDIES",         {"primary":True}),
    # Trials testing drugs
    ("T001","DR002","TESTS",          {"arm":"treatment"}),
    ("T001","DR001","TESTS",          {"arm":"comparator"}),
    ("T002","DR003","TESTS",          {"arm":"treatment"}),
    # Drugs treating diseases
    ("DR001","D001","TREATS",         {"evidence":"strong"}),
    ("DR002","D002","TREATS",         {"evidence":"strong"}),
    ("DR002","D003","TREATS",         {"evidence":"moderate"}),
    ("DR003","D002","TREATS",         {"evidence":"moderate"}),
    # Researcher collaborations
    ("R001","R004","COLLABORATES_WITH",{"papers":3}),
    ("R002","R003","COLLABORATES_WITH",{"papers":1}),
]
for src, dst, rel_type, props in edges:
    G.add_edge(src, dst, rel_type=rel_type, **props)

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
label_counts = {}
for n, d in G.nodes(data=True):
    label_counts[d['label']] = label_counts.get(d['label'], 0) + 1
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count}")
rel_counts = {}
for u, v, d in G.edges(data=True):
    rel_counts[d['rel_type']] = rel_counts.get(d['rel_type'], 0) + 1
print("Relationship types:")
for rel, count in sorted(rel_counts.items()):
    print(f"  {rel}: {count}")


Graph loaded: 15 nodes, 21 edges
  Disease: 3
  Drug: 3
  Institution: 3
  Researcher: 4
  Trial: 2
Relationship types:
  AFFILIATED_WITH: 4
  COLLABORATES_WITH: 2
  CONTRIBUTES_TO: 3
  LEADS: 2
  STUDIES: 3
  TESTS: 3
  TREATS: 4


---
## Nodes, relationships, and graph vs relational model

In [2]:
print("=== Graph data model: nodes, relationships, properties ===")
print()

print("NODES (equivalent to rows in a table, but schema-free):")
node_examples = [
    ("Researcher", "R001", {"name": "Aroha Ngata", "role": "PI", "h_index": 12}),
    ("Drug",       "DR001",{"name": "Metformin", "drug_class": "Biguanide", "approved": 1995}),
    ("Trial",      "T001", {"title": "Cardio Alpha", "phase": 2, "status": "active"}),
]
for label, node_id, props in node_examples:
    print(f"  (:{label} {{id: '{node_id}', {', '.join(f'{k}: {v!r}' for k,v in props.items())}}})")

print()
print("RELATIONSHIPS (directed, typed, can have properties):")
rel_examples = [
    ("R001", "LEADS",      "T001", {"role": "Principal Investigator", "since": "2023-01-01"}),
    ("T001", "TESTS",      "DR001",{"arm": "treatment", "dose": "500mg"}),
    ("DR001","TREATS",     "D001", {"evidence": "strong", "mechanism": "insulin_sensitizer"}),
]
for src, rel_type, dst, props in rel_examples:
    print(f"  ({src})-[:{rel_type} {{{', '.join(f'{k}: {v!r}' for k,v in props.items())}}}]->({dst})")

print()
print("=== Graph vs Relational model comparison ===")
comparison = [
    ("Data unit",          "Node (properties bag)",       "Row (fixed schema)"),
    ("Connection",         "Typed, directed relationship","Foreign key + JOIN"),
    ("Connection cost",    "O(1) pointer traversal",     "O(log N) index lookup"),
    ("Schema",             "Optional (labels + props)",  "Rigid (columns, types)"),
    ("Variable depth",     "Native (*1..n paths)",       "Recursive CTE needed"),
    ("Query language",     "Cypher (pattern matching)",  "SQL (set algebra)"),
    ("Aggregation",        "Supported but not primary",  "Primary strength"),
    ("Many-to-many",       "Direct edges both ways",     "Junction table"),
    ("Best for",           "Connected, traversal-heavy", "Tabular, aggregation-heavy"),
]
print(f"  {'Aspect':<22s}  {'Graph (Neo4j)':<35s}  Relational (PostgreSQL)")
print("  " + "-"*85)
for aspect, graph, rel in comparison:
    print(f"  {aspect:<22s}  {graph:<35s}  {rel}")


=== Graph data model: nodes, relationships, properties ===

NODES (equivalent to rows in a table, but schema-free):
  (:Researcher {id: 'R001', name: 'Aroha Ngata', role: 'PI', h_index: 12})
  (:Drug {id: 'DR001', name: 'Metformin', drug_class: 'Biguanide', approved: 1995})
  (:Trial {id: 'T001', title: 'Cardio Alpha', phase: 2, status: 'active'})

RELATIONSHIPS (directed, typed, can have properties):
  (R001)-[:LEADS {role: 'Principal Investigator', since: '2023-01-01'}]->(T001)
  (T001)-[:TESTS {arm: 'treatment', dose: '500mg'}]->(DR001)
  (DR001)-[:TREATS {evidence: 'strong', mechanism: 'insulin_sensitizer'}]->(D001)

=== Graph vs Relational model comparison ===
  Aspect                  Graph (Neo4j)                        Relational (PostgreSQL)
  -------------------------------------------------------------------------------------
  Data unit               Node (properties bag)                Row (fixed schema)
  Connection              Typed, directed relationship         Foreig

---
## When to choose a graph database

In [3]:
print("=== When to choose a graph database ===")
print()

use_cases = [
    ("Social networks",          "Friend-of-friend; influence; community detection"),
    ("Knowledge graphs",         "Entity relationships; semantic search; ontologies"),
    ("Fraud detection",          "Transaction rings; account linkage; suspicious patterns"),
    ("Recommendation engines",   "Collaborative filtering via graph traversal"),
    ("Clinical research networks","Researcher-trial-drug-disease connections (this demo)"),
    ("Supply chain",             "Dependency chains; bottleneck identification"),
    ("Identity/access graphs",   "Permission inheritance; role hierarchy"),
    ("Drug interaction mapping", "Drug-protein-pathway interaction networks"),
]
print("Graph databases excel at:")
for use_case, why in use_cases:
    print(f"  {use_case:<30s}  {why}")

print()
not_graph = [
    ("Bulk analytics / aggregation",  "SQL: GROUP BY, window functions, aggregates"),
    ("Regular tabular reporting",     "SQL: joins across well-defined tables"),
    ("Time series data",              "TimescaleDB, InfluxDB, or partitioned SQL tables"),
    ("Full-text search",              "PostgreSQL tsvector, Elasticsearch"),
    ("Simple CRUD applications",      "PostgreSQL or SQLite — no graph needed"),
]
print("Graph databases are NOT the right choice for:")
for case, alternative in not_graph:
    print(f"  {case:<34s}  -> {alternative}")

print()
print("Neo4j + Python connection patterns:")
conn_code = """
# neo4j-driver (official):
from neo4j import GraphDatabase
driver = GraphDatabase.driver("bolt://localhost:7687",
                               auth=("neo4j", "password"))
with driver.session() as session:
    result = session.run("MATCH (r:Researcher) RETURN r.name LIMIT 5")
    for record in result:
        print(record["r.name"])
driver.close()

# py2neo (higher-level):
from py2neo import Graph
graph = Graph("bolt://localhost:7687", auth=("neo4j", "password"))
results = graph.run("MATCH (r:Researcher) RETURN r.name LIMIT 5")
for row in results:
    print(row["r.name"])
"""
print(conn_code)


=== When to choose a graph database ===

Graph databases excel at:
  Social networks                 Friend-of-friend; influence; community detection
  Knowledge graphs                Entity relationships; semantic search; ontologies
  Fraud detection                 Transaction rings; account linkage; suspicious patterns
  Recommendation engines          Collaborative filtering via graph traversal
  Clinical research networks      Researcher-trial-drug-disease connections (this demo)
  Supply chain                    Dependency chains; bottleneck identification
  Identity/access graphs          Permission inheritance; role hierarchy
  Drug interaction mapping        Drug-protein-pathway interaction networks

Graph databases are NOT the right choice for:
  Bulk analytics / aggregation        -> SQL: GROUP BY, window functions, aggregates
  Regular tabular reporting           -> SQL: joins across well-defined tables
  Time series data                    -> TimescaleDB, InfluxDB, or part

---
## Common Pitfalls

**1. Modelling every relationship as a node property instead of an edge**
Storing `collaborators: ['R002', 'R004']` as a list property on a Researcher node misses the entire point of a graph. Each collaborator relationship should be an edge `(R001)-[:COLLABORATES_WITH]->(R002)`. Edge properties (date, number of papers) are searchable and traversable; list properties are not.

**2. Using graph databases for simple tabular data**
If your queries are mostly aggregations (`COUNT`, `AVG`, `SUM` by category) with no traversal, a graph database adds complexity without benefit. PostgreSQL with proper indexes is simpler, faster, and easier to maintain for tabular analytics.

**3. Not understanding that relationships are directed**
Neo4j relationships have a direction. `(R001)-[:LEADS]->(T001)` is not the same as `(T001)-[:LEADS]->(R001)`. In Cypher, you can query ignoring direction with `-[:LEADS]-` (no arrow), but the data is stored directed. Model direction to match the domain: a researcher LEADS a trial, not the reverse.

**4. Expecting relational aggregation performance from a graph**
Counting all nodes, running GROUP BY across millions of nodes, or doing full-graph scans for aggregate statistics is slower in Neo4j than in PostgreSQL. Graph databases are optimised for traversal, not bulk aggregation. For analytical reporting, sync data to a columnar store or use Cypher aggregation carefully.

**5. Not using MERGE for data loading pipelines**
Running `CREATE (r:Researcher {id: 'R001'})` twice creates duplicate nodes. In data loading and ETL pipelines, always use `MERGE (r:Researcher {id: $id})` which matches an existing node or creates it if absent. Create a uniqueness constraint on the merge key.

**6. No constraint on merge key — slow MERGE and duplicates**
`MERGE (r:Researcher {id: $id})` without a uniqueness constraint does a full label scan to find the node. `CREATE CONSTRAINT ON (r:Researcher) ASSERT r.id IS UNIQUE` makes MERGE an O(1) index lookup and prevents duplicates.


---
*sql_methods_library - Samantha McGarrigle*